# Sentiment Analysis Pipeline

## How to run

**To reproduce the final results (no API keys needed):** run all cells, then call:
```python
run_pipeline(load_cached_corpus="raw_corpus_cache.csv")
```
This skips data collection and uses the cached corpus committed to this repo. The output (`sentiment_features.csv`) is also already in the repo and is what the LSTM and MAB notebooks consume.

**To re-pull fresh data from NewsAPI / Reddit / GDELT:** set these environment variables before running:
- `NEWS_API_KEY`  — newsapi.org free account
- `REDDIT_CLIENT_ID` and `REDDIT_CLIENT_SECRET`  — from reddit.com/prefs/apps (script app)

Then call `run_pipeline()` with no `load_cached_corpus` argument.


In [1]:
!pip install transformers torch requests praw pandas numpy tqdm gdeltdoc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 33.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.2 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.


In [2]:
import os
import time
import requests
import praw
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import torch
from transformers import BertTokenizer, BertForSequenceClassification
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("praw").setLevel(logging.CRITICAL)

In [ ]:
import os

# API credentials are loaded from environment variables.
# To re-pull data, set these in your environment (or in Colab via
#   from google.colab import userdata; os.environ['NEWS_API_KEY'] = userdata.get('NEWS_API_KEY')
# ).
# To reproduce results without API access, see the README — pass
# load_cached_corpus="raw_corpus_cache.csv" into run_pipeline().
NEWS_API_KEY         = os.environ.get("NEWS_API_KEY", "")
REDDIT_CLIENT_ID     = os.environ.get("REDDIT_CLIENT_ID", "")
REDDIT_CLIENT_SECRET = os.environ.get("REDDIT_CLIENT_SECRET", "")
REDDIT_USER_AGENT    = "energy_sentiment_bot/0.1"


In [4]:
TICKERS = ["XOM", "CVX", "COP", "OXY", "PXD", "SLB", "EOG", "DVN", "MPC"]

TICKER_KEYWORDS = {
    "XOM": ["XOM", "Exxon", "ExxonMobil"],
    "CVX": ["CVX", "Chevron"],
    "COP": ["COP", "ConocoPhillips", "Conoco"],
    "OXY": ["OXY", "Occidental", "Occidental Petroleum"],
    "PXD": ["PXD", "Pioneer", "Pioneer Natural Resources"],
    "SLB": ["SLB", "Schlumberger"],
    "EOG": ["EOG", "EOG Resources"],
    "DVN": ["DVN", "Devon", "Devon Energy"],
    "MPC": ["MPC", "Marathon Petroleum"],
}

START_DATE = "2020-01-01"
END_DATE   = "2026-04-30"

# ── Market close cutoff (ET)
MARKET_CLOSE_HOUR = 16  # 4 PM ET — items after this → next trading day

In [5]:
# NewsAPI Collector
# NOTE: Free tier only gives ~1 month back. For full history,
# use GDELT (Cell 5) which covers 2020–present for free.
class NewsAPICollector:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://newsapi.org/v2/everything"

    def fetch_ticker(self, ticker, keywords, from_date, to_date):
        """Fetch headlines for one ticker. Returns list of dicts."""
        query = " OR ".join(f'"{kw}"' for kw in keywords)
        params = {
            "q": query,
            "from": from_date,
            "to": to_date,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 100,
            "apiKey": self.api_key,
        }
        results = []
        page = 1
        while True:
            params["page"] = page
            resp = requests.get(self.base_url, params=params)
            if resp.status_code != 200:
                print(f"  NewsAPI error {resp.status_code} for {ticker}")
                break
            data = resp.json()
            articles = data.get("articles", [])
            if not articles:
                break
            for art in articles:
                results.append({
                    "ticker": ticker,
                    "source": "newsapi",
                    "text": f"{art.get('title','')} {art.get('description','')}".strip(),
                    "published_at": art.get("publishedAt", ""),
                })
            if len(articles) < 100:
                break
            page += 1
            time.sleep(0.3)  # rate limit courtesy
        return results

    def fetch_all(self, from_date, to_date):
        all_rows = []
        for ticker, keywords in TICKER_KEYWORDS.items():
            print(f"  NewsAPI → {ticker}")
            rows = self.fetch_ticker(ticker, keywords, from_date, to_date)
            all_rows.extend(rows)
            time.sleep(1)
        return pd.DataFrame(all_rows)

In [6]:
# Reddit Collector (PRAW — fully free)
# Subreddits: r/investing, r/stocks, r/wallstreetbets, r/energy

class RedditCollector:
    def __init__(self, client_id, client_secret, user_agent):
        self.reddit = praw.Reddit(
            client_id=client_id,
            client_secret=client_secret,
            user_agent=user_agent,
        )
        self.subreddits = ["investing", "stocks", "wallstreetbets", "energy", "StockMarket"]

    def _matches_ticker(self, text, ticker):
        keywords = TICKER_KEYWORDS[ticker]
        text_lower = text.lower()
        return any(kw.lower() in text_lower for kw in keywords)

    def fetch_subreddit(self, subreddit_name, limit=1000):
        """Pull top + new posts from a subreddit."""
        rows = []
        sub = self.reddit.subreddit(subreddit_name)
        for post in sub.search("energy OR oil OR gas OR crude", limit=limit, sort="new"):
            text = f"{post.title} {post.selftext}"
            published_at = datetime.utcfromtimestamp(post.created_utc).isoformat()
            for ticker in TICKERS:
                if self._matches_ticker(text, ticker):
                    rows.append({
                        "ticker": ticker,
                        "source": f"reddit_{subreddit_name}",
                        "text": post.title,   # title only — cleaner signal
                        "published_at": published_at,
                    })
        return rows

    def fetch_all(self):
        all_rows = []
        for sub in self.subreddits:
            print(f"  Reddit → r/{sub}")
            try:
                rows = self.fetch_subreddit(sub)
                all_rows.extend(rows)
                time.sleep(2)
            except Exception as e:
                print(f"    Error on r/{sub}: {e}")
        return pd.DataFrame(all_rows)

In [7]:
# GDELT Collector (completely free, covers 2015–now)
# Best free source for historical news coverage

class GDELTCollector:
    """
    Uses GDELT's pre-built bulk CSV files instead of the live API.
    No rate limits. Downloads directly from Google Cloud Storage.
    Filters for energy ticker mentions locally.
    """
    # GDELT exports a master file list updated every 15 minutes
    MASTER_URL = "http://data.gdeltproject.org/gdeltv2/masterfilelist.txt"

    def _download_chunk(self, url):
        """Download and parse one GDELT 15-min CSV chunk."""
        import io, zipfile
        try:
            resp = requests.get(url, timeout=60)
            if resp.status_code != 200:
                return pd.DataFrame()
            z = zipfile.ZipFile(io.BytesIO(resp.content))
            fname = z.namelist()[0]
            with z.open(fname) as f:
                # GDELT GKG has 27 columns — we only need a few
                df = pd.read_csv(f, sep="\t", header=None,
                                 usecols=[0, 4],  # DATE, SourceCommonName, Themes, Locations, Persons, Orgs, Tone
                                 names=["date", "themes"],
                                 on_bad_lines="skip",
                                 encoding_errors="ignore")
            return df
        except:
            return pd.DataFrame()

    def fetch_from_gdelt_doc(self, start_date, end_date):
        """
        Use GDELT DOC 2.0 API but ONE query per day (not per ticker per month).
        Broader query = fewer requests = fewer rate limits.
        """
        all_rows = []

        # Build one broad energy query instead of 10 ticker queries
        energy_query = (
            '"Exxon" OR "Chevron" OR "ConocoPhillips" OR "Occidental" '
            'OR "Pioneer Natural" OR "Schlumberger" OR "Halliburton" '
            'OR "EOG Resources" OR "Devon Energy" OR "Marathon Petroleum"'
        )

        start = datetime.strptime(start_date, "%Y-%m-%d")
        end   = datetime.strptime(end_date,   "%Y-%m-%d")
        current = start

        while current < end:
            chunk_end = min(current + timedelta(days=90), end)  # 3-month chunks
            cs = current.strftime("%Y-%m-%d")
            ce = chunk_end.strftime("%Y-%m-%d")

            params = {
                "query": f"({energy_query}) sourcelang:english",
                "mode": "artlist",
                "maxrecords": 250,
                "startdatetime": current.strftime("%Y%m%d") + "000000",
                "enddatetime":   chunk_end.strftime("%Y%m%d") + "235959",
                "format": "json",
            }

            print(f"  GDELT → energy basket | {cs} to {ce}")
            success = False

            for attempt in range(5):
                try:
                    time.sleep(10 + attempt * 10)  # 10s, 20s, 30s... before each attempt
                    resp = requests.get(
                        "https://api.gdeltproject.org/api/v2/doc/doc",
                        params=params, timeout=45
                    )
                    if resp.status_code == 429 or not resp.text.strip():
                        print(f"    Retry {attempt+1}/5 — waiting {30*(attempt+1)}s")
                        time.sleep(30 * (attempt + 1))
                        continue

                    data = resp.json()
                    articles = data.get("articles", [])

                    # Now assign tickers locally by scanning each title
                    for art in articles:
                        title = art.get("title", "")
                        pub   = art.get("seendate", "")
                        for ticker, keywords in TICKER_KEYWORDS.items():
                            if any(kw.lower() in title.lower() for kw in keywords):
                                all_rows.append({
                                    "ticker": ticker,
                                    "source": "gdelt",
                                    "text": title,
                                    "published_at": pub,
                                })

                    print(f"    Got {len(articles)} articles")
                    success = True
                    break

                except Exception as e:
                    print(f"    Exception attempt {attempt+1}: {e}")
                    time.sleep(30)

            if not success:
                print(f"    Skipping {cs}–{ce} after all retries")

            current = chunk_end + timedelta(days=1)
            time.sleep(20)  # 20s between chunks

        return pd.DataFrame(all_rows) if all_rows else pd.DataFrame(
            columns=["ticker","source","text","published_at"]
        )

In [8]:
def clean_text(text):
    """Basic cleaning before FinBERT tokenization."""
    import re
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+", "", text)          # remove URLs
    text = re.sub(r"[$#@]\w+", "", text)         # remove cashtags/hashtags
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'-]", "", text)  # strip special chars
    text = re.sub(r"\s+", " ", text).strip()
    return text

def assign_trading_day(published_at_str):
    """
    Converts a timestamp string to a trading day.
    Items published after 4 PM ET → assigned to next calendar day.
    (Weekends/holidays are handled later during merge with price data.)
    """
    try:
        # Handle both ISO format and GDELT format (YYYYMMDDTHHMMSSZ)
        ts_str = str(published_at_str)
        if "T" in ts_str and len(ts_str) > 15:
            # GDELT format: 20220415T143000Z
            dt = datetime.strptime(ts_str[:15], "%Y%m%dT%H%M%S")
        else:
            dt = pd.to_datetime(ts_str, utc=True).tz_localize(None)
        # Shift items published after market close to next day
        if dt.hour >= MARKET_CLOSE_HOUR:
            dt = dt + timedelta(days=1)
        return dt.date()
    except:
        return None

def build_raw_corpus(news_df, reddit_df, gdelt_df):
    """Combine all sources, clean text, assign trading days."""
    combined = pd.concat([news_df, reddit_df, gdelt_df], ignore_index=True)
    combined["text"] = combined["text"].apply(clean_text)

    # Drop empties and very short texts
    combined = combined[combined["text"].str.len() > 20].copy()

    # Assign trading day
    combined["trading_day"] = combined["published_at"].apply(assign_trading_day)
    combined = combined.dropna(subset=["trading_day"])

    # Deduplicate near-identical headlines per ticker per day
    combined = combined.drop_duplicates(subset=["ticker", "trading_day", "text"])

    print(f"\nRaw corpus: {len(combined):,} items across {combined['ticker'].nunique()} tickers")
    print(combined.groupby("ticker").size().to_string())
    return combined

In [9]:
# FinBERT Inference

class FinBERTScorer:
    def __init__(self, batch_size=32, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading FinBERT on {self.device}...")
        self.tokenizer = BertTokenizer.from_pretrained("ProsusAI/finbert")
        self.model = BertForSequenceClassification.from_pretrained("ProsusAI/finbert")
        self.model.to(self.device)
        self.model.eval()
        self.batch_size = batch_size
        # FinBERT label order: positive=0, negative=1, neutral=2
        self.labels = ["positive", "negative", "neutral"]

    def score_batch(self, texts):
        """Returns array of shape (N, 3): [P_pos, P_neg, P_neutral]"""
        all_probs = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i : i + self.batch_size]
            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=128,   # headlines rarely exceed 128 tokens
                return_tensors="pt",
            ).to(self.device)
            with torch.no_grad():
                logits = self.model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)  # shape: (N, 3)

    def score_dataframe(self, df):
        """Adds p_positive, p_negative, p_neutral columns to df."""
        texts = df["text"].tolist()
        print(f"Running FinBERT on {len(texts):,} items...")
        probs = self.score_batch(texts)
        df = df.copy()
        df["p_positive"] = probs[:, 0]
        df["p_negative"] = probs[:, 1]
        df["p_neutral"]  = probs[:, 2]
        df["net_sentiment"] = df["p_positive"] - df["p_negative"]
        return df

In [10]:
# Daily Aggregation → Sentiment Feature Tensor

def aggregate_daily_sentiment(scored_df):
    """
    Collapses per-item FinBERT scores to a daily feature vector per ticker.

    Output columns per (ticker, trading_day):
      - p_positive_mean   : mean positive probability
      - p_negative_mean   : mean negative probability
      - p_neutral_mean    : mean neutral probability
      - net_sentiment     : mean(P_pos - P_neg)
      - sentiment_volume  : log(1 + item count)  ← captures attention level
      - sentiment_momentum: 3-day rolling mean of net_sentiment
      - sentiment_dispersion: std of net_sentiment across items that day
    """
    grp = scored_df.groupby(["ticker", "trading_day"])

    agg = grp.agg(
        p_positive_mean   = ("p_positive",    "mean"),
        p_negative_mean   = ("p_negative",    "mean"),
        p_neutral_mean    = ("p_neutral",     "mean"),
        net_sentiment     = ("net_sentiment", "mean"),
        item_count        = ("text",          "count"),
        sentiment_dispersion = ("net_sentiment", "std"),
    ).reset_index()

    # Log-scale volume so outlier days don't dominate
    agg["sentiment_volume"] = np.log1p(agg["item_count"])
    agg["sentiment_dispersion"] = agg["sentiment_dispersion"].fillna(0)

    # 3-day rolling sentiment momentum (per ticker)
    agg = agg.sort_values(["ticker", "trading_day"])
    agg["sentiment_momentum"] = (
        agg.groupby("ticker")["net_sentiment"]
           .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

    # Final 7-dim feature vector columns
    feature_cols = [
        "p_positive_mean", "p_negative_mean", "p_neutral_mean",
        "net_sentiment", "sentiment_volume",
        "sentiment_momentum", "sentiment_dispersion",
    ]

    print(f"\nAggregated sentiment: {len(agg):,} (ticker, day) pairs")
    print(f"Date range: {agg['trading_day'].min()} → {agg['trading_day'].max()}")
    return agg[["ticker", "trading_day"] + feature_cols]

In [11]:
# Normalize (training window only — no lookahead)

TRAIN_START = "2022-01-01"
TRAIN_END   = "2024-12-31"

FEATURE_COLS = [
    "p_positive_mean", "p_negative_mean", "p_neutral_mean",
    "net_sentiment", "sentiment_volume",
    "sentiment_momentum", "sentiment_dispersion",
]

def normalize_sentiment_features(daily_df):
    """
    Z-score normalization using ONLY training window statistics.
    Applies the same mean/std to validation and test sets.
    Returns normalized df + the scaler stats (save these for inference).
    """
    train_mask = (
        (daily_df["trading_day"] >= pd.to_datetime(TRAIN_START).date()) &
        (daily_df["trading_day"] <= pd.to_datetime(TRAIN_END).date())
    )
    train_data = daily_df[train_mask]

    scaler_stats = {}
    for col in FEATURE_COLS:
        mu  = train_data[col].mean()
        std = train_data[col].std()
        std = std if std > 1e-8 else 1.0   # avoid divide-by-zero
        scaler_stats[col] = {"mean": mu, "std": std}

    df_norm = daily_df.copy()
    for col in FEATURE_COLS:
        mu  = scaler_stats[col]["mean"]
        std = scaler_stats[col]["std"]
        df_norm[col] = (df_norm[col] - mu) / std

    print("\nNormalization stats (training window):")
    stats_df = pd.DataFrame(scaler_stats).T
    print(stats_df.round(4).to_string())

    return df_norm, scaler_stats

In [12]:
# Fill Missing Trading Days

def fill_missing_days(daily_df, trading_calendar_df=None):
    """
    Ensure every ticker has an entry for every trading day.
    Missing days → forward-fill sentiment, zero volume.

    Pass in a DataFrame with a 'trading_day' column representing
    the full calendar (can derive from your price data index).
    If not provided, uses business days as an approximation.
    """
    if trading_calendar_df is None:
        all_days = pd.bdate_range(start=START_DATE, end=END_DATE).date
    else:
        all_days = trading_calendar_df["trading_day"].values

    filled_frames = []
    for ticker in TICKERS:
        ticker_df = daily_df[daily_df["ticker"] == ticker].copy()
        ticker_df = ticker_df.set_index("trading_day")

        full_index = pd.Index(all_days, name="trading_day")
        ticker_df  = ticker_df.reindex(full_index)
        ticker_df["ticker"] = ticker

        # Forward-fill sentiment; missing volume → 0 (no news that day)
        sentiment_cols = [c for c in FEATURE_COLS if c != "sentiment_volume"]
        ticker_df[sentiment_cols] = ticker_df[sentiment_cols].ffill()
        ticker_df["sentiment_volume"] = ticker_df["sentiment_volume"].fillna(0)

        # Any remaining NaN at the very start → neutral (0 after z-score ≈ fill with 0)
        ticker_df[FEATURE_COLS] = ticker_df[FEATURE_COLS].fillna(0)
        ticker_df = ticker_df.reset_index()
        filled_frames.append(ticker_df)

    result = pd.concat(filled_frames, ignore_index=True)
    print(f"\nAfter filling: {len(result):,} rows ({len(TICKERS)} tickers × {len(all_days)} days)")
    return result

In [13]:
# Master Pipeline Runner

def run_pipeline(
    news_api_key=NEWS_API_KEY,
    reddit_creds=(REDDIT_CLIENT_ID, REDDIT_CLIENT_SECRET, REDDIT_USER_AGENT),
    start_date=START_DATE,
    end_date=END_DATE,
    output_path="sentiment_features.csv",
    load_cached_corpus=None,   # pass a .csv path to skip collection
):
    """
    End-to-end pipeline:
      1. Collect from NewsAPI + Reddit + GDELT
      2. Clean & assign trading days
      3. FinBERT scoring
      4. Daily aggregation
      5. Normalize on training window
      6. Fill missing days
      7. Save final tensor as CSV
    """

    # ── Step 1: Collect ─────────────────────────────────────
    if load_cached_corpus:
        print(f"Loading cached corpus from {load_cached_corpus}")
        corpus = pd.read_csv(load_cached_corpus)
    else:
        print("=== Step 1: Collecting data ===")

        # NewsAPI (recent ~1 month on free tier)
        print("\n[NewsAPI]")
        recent_start = (datetime.now() - timedelta(days=28)).strftime("%Y-%m-%d")
        news_collector = NewsAPICollector(news_api_key)
        news_df = news_collector.fetch_all(recent_start, end_date)
        print(f"  → {len(news_df)} articles")

        # Reddit
        print("\n[Reddit]")
        reddit_collector = RedditCollector(*reddit_creds)
        reddit_df = reddit_collector.fetch_all()
        print(f"  → {len(reddit_df)} posts")

        # GDELT (full historical range — chunked by month)
        print("\n[GDELT — this will take a few minutes]")
        gdelt_collector = GDELTCollector()
        gdelt_df = gdelt_collector.fetch_from_gdelt_doc(start_date, end_date)
        print(f"  → {len(gdelt_df)} articles")

        # ── Step 2: Clean & combine
        print("\n=== Step 2: Cleaning & combining ===")
        corpus = build_raw_corpus(news_df, reddit_df, gdelt_df)
        corpus.to_csv("raw_corpus_cache.csv", index=False)
        print("Raw corpus saved to raw_corpus_cache.csv")

    # ── Step 3: FinBERT scoring
    print("\n=== Step 3: FinBERT scoring ===")
    scorer = FinBERTScorer(batch_size=64)
    scored = scorer.score_dataframe(corpus)

    # ── Step 4: Daily aggregation
    print("\n=== Step 4: Daily aggregation ===")
    daily = aggregate_daily_sentiment(scored)

    # ── Step 5: Normalize
    print("\n=== Step 5: Normalizing (train window only) ===")
    daily_norm, scaler_stats = normalize_sentiment_features(daily)

    # Save scaler stats for later use during inference
    pd.DataFrame(scaler_stats).T.to_csv("sentiment_scaler_stats.csv")
    print("Scaler stats saved to sentiment_scaler_stats.csv")

    # ── Step 6: Fill missing days
    print("\n=== Step 6: Filling missing trading days ===")
    final = fill_missing_days(daily_norm)

    # ── Step 7: Save
    final.to_csv(output_path, index=False)
    print(f"\n✓ Final sentiment tensor saved to: {output_path}")
    print(f"  Shape: {final.shape}  →  (ticker-days × features)")
    print(f"  Columns: {list(final.columns)}")

    return final, scaler_stats

In [14]:
# Sanity Checks

def sanity_check(final_df):
    """
    Spot-check the sentiment output against known market events.
    Good test: OXY in Aug 2022 (Berkshire acquisition news)
               XOM in March 2020 (COVID crash)
    """
    print("=== Sanity Checks ===\n")

    # 1. No NaNs in feature columns
    nan_count = final_df[FEATURE_COLS].isna().sum().sum()
    print(f"NaN count in features: {nan_count}  (should be 0)")

    # 2. Net sentiment range
    ns = final_df["net_sentiment"]
    print(f"net_sentiment range: [{ns.min():.3f}, {ns.max():.3f}]  (z-scored, ~[-3,3] expected)")

    # 3. Spot-check: XOM around March 2020 (should see high negative sentiment)
    xom = final_df[
        (final_df["ticker"] == "XOM") &
        (final_df["trading_day"].astype(str) >= "2020-03-01") &
        (final_df["trading_day"].astype(str) <= "2020-03-31")
    ]
    if not xom.empty:
        print(f"\nXOM March 2020 — mean net_sentiment: {xom['net_sentiment'].mean():.3f}  (expect negative)")
    else:
        print("\nXOM March 2020 — no data (check GDELT coverage)")

    # 4. Spot-check: OXY Aug 2022 (Berkshire acquisition — expect positive)
    oxy = final_df[
        (final_df["ticker"] == "OXY") &
        (final_df["trading_day"].astype(str) >= "2022-08-01") &
        (final_df["trading_day"].astype(str) <= "2022-08-31")
    ]
    if not oxy.empty:
        print(f"OXY Aug 2022 — mean net_sentiment: {oxy['net_sentiment'].mean():.3f}  (expect positive)")

    # 5. Volume distribution
    print(f"\nMedian daily item count per ticker:")
    vol = final_df.groupby("ticker")["sentiment_volume"].median()
    print(vol.to_string())

In [ ]:
# Run Everything

if __name__ == "__main__":
    # ── Option A: Full run (first time)
    final_df, scaler_stats = run_pipeline(
        output_path="sentiment_features.csv",
        # load_cached_corpus="raw_corpus_cache.csv",  # uncomment to skip re-collection
    )

    # ── Sanity checks
    sanity_check(final_df)

    # ── Quick peek
    print("\nSample output (first 5 rows):")
    print(final_df.head().to_string())

=== Step 1: Collecting data ===

[NewsAPI]
  NewsAPI → XOM
  NewsAPI → CVX
  NewsAPI → COP
  NewsAPI → OXY
  NewsAPI → PXD
  NewsAPI → SLB
  NewsAPI → EOG
  NewsAPI → DVN
  NewsAPI → MPC
  → 736 articles

[Reddit]
  Reddit → r/investing
  Reddit → r/stocks
  Reddit → r/wallstreetbets
  Reddit → r/energy
  Reddit → r/StockMarket
  → 143 posts

[GDELT — this will take a few minutes]
  GDELT → energy basket | 2020-01-01 to 2020-03-31
    Got 250 articles
  GDELT → energy basket | 2020-04-01 to 2020-06-30
    Got 250 articles
  GDELT → energy basket | 2020-07-01 to 2020-09-29
    Retry 1/5 — waiting 30s
    Retry 2/5 — waiting 60s
    Retry 3/5 — waiting 90s
    Retry 4/5 — waiting 120s
    Retry 5/5 — waiting 150s
    Skipping 2020-07-01–2020-09-29 after all retries
  GDELT → energy basket | 2020-09-30 to 2020-12-29
    Got 250 articles
  GDELT → energy basket | 2020-12-30 to 2021-03-30
    Retry 1/5 — waiting 30s
    Retry 2/5 — waiting 60s
    Got 250 articles
  GDELT → energy basket | 

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running FinBERT on 2,375 items...

=== Step 4: Daily aggregation ===

Aggregated sentiment: 1,472 (ticker, day) pairs
Date range: 2020-01-05 → 2026-04-30

=== Step 5: Normalizing (train window only) ===

Normalization stats (training window):
                        mean     std
p_positive_mean       0.2516  0.2914
p_negative_mean       0.2000  0.3000
p_neutral_mean        0.5484  0.3397
net_sentiment         0.0517  0.4842
sentiment_volume      0.8884  0.3820
sentiment_momentum    0.0539  0.3128
sentiment_dispersion  0.0947  0.2137
Scaler stats saved to sentiment_scaler_stats.csv

=== Step 6: Filling missing trading days ===

After filling: 14,868 rows (9 tickers × 1652 days)

✓ Final sentiment tensor saved to: sentiment_features.csv
  Shape: (14868, 9)  →  (ticker-days × features)
  Columns: ['trading_day', 'ticker', 'p_positive_mean', 'p_negative_mean', 'p_neutral_mean', 'net_sentiment', 'sentiment_volume', 'sentiment_momentum', 'sentiment_dispersion']
=== Sanity Checks ===

NaN cou

In [ ]:
def verify_sources(corpus_df):
    print("=== SOURCE VERIFICATION ===\n")

    # 1. Count by source
    print("Items per source:")
    print(corpus_df["source"].value_counts().to_string())

    # 2. Count by source AND ticker
    print("\nItems per source per ticker:")
    pivot = corpus_df.groupby(["source", "ticker"]).size().unstack(fill_value=0)
    print(pivot.to_string())

    # 3. Date range per source
    print("\nDate range per source:")
    corpus_df["trading_day"] = pd.to_datetime(corpus_df["trading_day"])
    for source in corpus_df["source"].unique():
        sub = corpus_df[corpus_df["source"] == source]
        print(f"  {source}: {sub['trading_day'].min().date()} → {sub['trading_day'].max().date()}  ({len(sub)} items)")

    # 4. Flag if any source is missing entirely
    print("\nSource health check:")
    expected = {"newsapi", "gdelt"}
    # Reddit has multiple source labels (reddit_investing, reddit_stocks etc.)
    reddit_sources = [s for s in corpus_df["source"].unique() if "reddit" in s.lower()]
    if reddit_sources:
        expected.add("reddit")

    found = set()
    for s in corpus_df["source"].unique():
        if "reddit" in s.lower():
            found.add("reddit")
        else:
            found.add(s)

    missing = expected - found
    if missing:
        print(f"  ⚠️  MISSING SOURCES: {missing}")
    else:
        print(f"  ✓ All sources present: {found}")

    # 5. What % of ticker-days have at least one item from each source
    print("\n% of trading days with coverage per source:")
    all_days = corpus_df["trading_day"].nunique()
    for source in corpus_df["source"].unique():
        days_covered = corpus_df[corpus_df["source"] == source]["trading_day"].nunique()
        print(f"  {source}: {days_covered}/{all_days} days ({100*days_covered/all_days:.1f}%)")

verify_sources(corpus)

NameError: name 'corpus' is not defined